In [ ]:
import pandas as pd
import os
from datetime import datetime, timedelta

## Compile Dataset Methods

In [ ]:
"""
Check to see if inputted dataset has more than 14 days by parsing data/time column in files

    Input: 
        orgDir - path of the directory where the original csv files are
        fileName - name of file being observed
    Return: fileName param, boolean, List
    boolean representing if number of days greater than 14 days and list of days that
    need to be removed
"""
def isFourteenDays(orgDir, fileName, newDirectory):
    # Load data
    filePath = os.path.join(orgDir, fileName)
    df = pd.read_csv(filePath)

    # Parse the date column (assuming it's named 'Date'). Adjust if the name is different.
    df['time'] = pd.to_datetime(df['time'])
    
    # Calculate date range
    min_date = df['time'].min()
    print(min_date)
    max_date = df['time'].max()    
    print(max_date)
    days_difference = (max_date - min_date).days
    print(days_difference)
    
    # Identify extra dates beyond the 14-day range
    if days_difference > 14:
        cutoff_date = min_date + timedelta(days=14)
        extra_days = df[df['time'] > cutoff_date]['time'].unique().tolist()
        return fileName, True, extra_days
    else:
        new_file_path = os.path.join(newDirectory, f"filtered_{os.path.basename(fileName)}")
        if not os.path.exists(new_file_path):
            df.to_csv(new_file_path, index=False)
        return fileName, False, []

"""
Using the method above, create a new file that will have the extra days removed from the set
    Input: orgDir, fileName, removeDays, newdirectory
        fileName - name of file beign observed
        removeDays - List of days that need to be removed
        newdirectory - the directory path where the new file will be stored
    Return: None
"""
def removeExtraDays(orgDir, fileName, removeDays, newdirectory):
    # Load the data
    orgDirPath = os.path.join(orgDir, fileName)
    df = pd.read_csv(orgDirPath)
    
    # Convert date column to datetime format for comparison
    df['time'] = pd.to_datetime(df['time'])
    
    # Remove the specified days from the dataset
    df_filtered = df[~df['time'].isin(pd.to_datetime(removeDays))]
    
    # Ensure the output directory exists
    if not os.path.exists(newdirectory):
        os.makedirs(newdirectory)
    
    # Save the filtered file in the new directory
    new_file_path = os.path.join(newdirectory, f"filtered_{os.path.basename(fileName)}")
    df_filtered.to_csv(new_file_path, index=False)

"""
Using the files in the directory, concat files into a new dataset. Save the new dataset inside
of the new directory
Make sure to add a new column that represents the patient ID before concatting files
    Input: oldDirectory, newDirectory
        oldDirectory - the directory path where the preprocessing dats is compiled
        newDirectory - the directory path where the concatted data will be stored
    oldDirectory and newDirectory are not going to be same so files will be easily differentiable
    Return: None
"""
##This method needs to be debugged. We may not need to oldDirectory parameter if we add the files
## that have at most 14 days into the new directory in isFouteenDays possibly
def concatFiles(oldDirectory, newDirectory):
    all_data = []
    patient_id = 1
    
    # Loop through each file in the old directory
    for file in os.listdir(oldDirectory):
        if file.endswith(".csv"):
            file_path = os.path.join(oldDirectory, file)
            
            # Load file data
            df = pd.read_csv(file_path)
            
            # Add PatientID column
            df['PatientID'] = patient_id
            all_data.append(df)
            
            # Increment PatientID for the next file
            patient_id += 1
    
    # Concatenate all dataframes in the list
    concatenated_df = pd.concat(all_data, ignore_index=True)
    print(concatenated_df)
    firstcol = concatenated_df.pop('PatientID')
    concatenated_df.insert(0, 'PatientID', firstcol)
    print(concatenated_df)
    # Ensure the output directory exists
    if not os.path.exists(newDirectory):
        os.makedirs(newDirectory)
    
    # Save the concatenated dataframe
    output_file = os.path.join(newDirectory, "concatenated_data.csv")
    concatenated_df.to_csv(output_file, index=False)

In [ ]:
def delete_files_in_directory(directory_path):
   try:
     files = os.listdir(directory_path)
     for file in files:
       file_path = os.path.join(directory_path, file)
       if os.path.isfile(file_path):
         os.remove(file_path)
     print("All files deleted successfully.")
   except OSError:
     print("Error occurred while deleting files.")

## TESTING

In [ ]:
##Getting original directory from preprocessing data
orgDataDir = os.path.join(os.getcwd(), "HUPA-UCM Diabetes Dataset")
orgDataDir = os.path.join(orgDataDir, 'Preprocessed')

print(orgDataDir)

##Getting the list of files within the directory
fileList = os.listdir(orgDataDir)
print(fileList)

##Getting the path of the directory where the new files will be stored
concatDataDir = os.path.join(os.getcwd(), "Final_Dataset")
filteredDataDir = os.path.join(concatDataDir, "Filtered_Data")
print(concatDataDir)
print(filteredDataDir)

#### Check files that have less than or equal to 14 dates

##### Need to test isFourteenDays()

In [ ]:
##Get the path of the file that needs to be opened
file1_name = "HUPA0002P.csv"

##Check to see whether this is 14 days worth of data
fileName, isfourteen, extraDaysList = isFourteenDays(orgDataDir, file1_name, filteredDataDir)
print("File Observed", fileName)
print("Does Dataset Contain Fourteen Days: ", isfourteen)
print("Extra Days", extraDaysList)

##

#### Check files that have greater than 14 dates
##### Test the functions isFourteenDays() and removeExtraDays()

In [ ]:
file2_name = 'HUPA0027P.csv'
fileName, isfourteen, extraDaysList = isFourteenDays(orgDataDir, file2_name, filteredDataDir)
print("##Check if 14 Days##")
print("File Observed", fileName)
print("Does Dataset Contain Fourteen Days: ", isfourteen)
print("Extra Days", extraDaysList, "\n")

print("##Remove Days Test##")
removeExtraDays(orgDataDir, file2_name, extraDaysList, filteredDataDir)

fileName, isfourteen, extraDaysList = isFourteenDays(orgDataDir, file2_name, filteredDataDir)
print("##Check if New File is 14 Days##")
print("File Observed", fileName)
print("Does Dataset Contain Fourteen Days: ", isfourteen)
print("Extra Days", extraDaysList, "\n")

#### Check if concatenation works properly

In [ ]:
concatFiles(filteredDataDir, concatDataDir)

In [ ]:
delete_files_in_directory(filteredDataDir)
delete_files_in_directory(concatDataDir)

## ADD SLEEP FEATURE TO ORIGINAL DATASET

In [ ]:
sleep_directory = os.path.join(os.getcwd(), "HUPA-UCM Diabetes Dataset")
sleep_directory = os.path.join(sleep_directory, 'Raw_Data')
sleep_dir_folders = os.listdir(sleep_directory)

night_files = {}
for dir in sleep_dir_folders:
    sleep_path = os.path.join(sleep_directory, dir, "fitbit")
    temp_list = []
    for file in os.listdir(sleep_path):
        if "_night" in file:
            temp_list.append(os.path.join(sleep_path, file))
    night_files[dir] = temp_list

c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-14_night.csv
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-14_night_summary.csv
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-15_night.csv
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-15_night_summary.csv
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-16_night.csv
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sle

In [48]:
data_checks = {}

##return number of wake states
def calculateAwakening(fileName):
    wake_state_count = 0
    data = pd.read_csv(fileName)
    print(data['State'], len(data['State']))
    for i in range(len(data['State'])):
        if (data['State'][i] == "wake"):
            wake_state_count += 1
    return wake_state_count

def efficiencyandWASOCalc(fileName):
    data = pd.read_csv(fileName)
    print(data)
    print(data['Efficiency'])
    print(data['Minutes Awake'])
    return data['Efficiency'], data['Minutes Awake']


In [49]:
file = night_files["HUPA0001P"][0]
awake_state_count = calculateAwakening(file)
file1 = night_files["HUPA0001P"][1]
efficiency, waso = efficiencyandWASOCalc(file1)
print(awake_state_count)
print(efficiency, waso)

0      wake
1     light
2      wake
3     light
4      deep
5     light
6       rem
7     light
8      deep
9     light
10     deep
11    light
12     wake
13      rem
14    light
15     wake
16    light
17      rem
18    light
19     deep
20    light
21     wake
22    light
23      rem
24    light
25     deep
26     wake
27    light
28     deep
29    light
30      rem
31    light
32     wake
Name: State, dtype: object 33
         Date               Start Time                 End Time  MainSleep  \
0  2018-06-14  2018-06-14T02:50:00.000  2018-06-14T10:06:00.000       True   

   Efficiency  Duration  Minutes Asleep  Minutes Light  Minutes Deep  \
0          94  26160000             390            226            60   

   Minutes REM  Minutes Awake  Minutes in Bed  
0          104             46             436  
0    94
Name: Efficiency, dtype: int64
0    46
Name: Minutes Awake, dtype: int64
7
0    94
Name: Efficiency, dtype: int64 0    46
Name: Minutes Awake, dtype: int64


### CALLING ALL METHODS TO CONCATENATE ALL FILES INTO THE FINAL_DATASET DIRECTORY